# Regulatory Network Analysis of adult Stem Cells from colon Xenium slide (480 probes) using scPRINT 

* **Author**: Anna Maguza 
* **Location**: CellZome, GSK company
* **Creation date**: 26.09.2025
* **Last modified date**: 26.09.2025

In [45]:
from huggingface_hub import hf_hub_download

from scprint import scPrint
from scdataloader import Preprocessor, utils
from scdataloader.preprocess import additional_preprocess
from scdataloader.utils import load_genes
from scprint.tasks import GNInfer
from grnndata import utils as grnutils
from grnndata import read_h5ad

import lamindb as ln
import scanpy as sc
import numpy as np
from pybiomart import Dataset
from matplotlib import pyplot as plt
import pandas as pd
import json
from anndata.utils import make_index_unique

import gseapy as gp
from gseapy import dotplot
from pyvis import network as pnx
import networkx as nx
import scipy.sparse

%load_ext autoreload
%autoreload 2 

import torch
torch.set_float32_matmul_precision('medium')

import os
import ssl
import urllib3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Preprocessing

In [ ]:
adata = sc.read_h5ad("/xenium_adult_colon_hs_NicheCompass_AM_26032025_173236_raw.h5ad")
adata

AnnData object with n_obs × n_vars = 274037 × 425
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'CXCL14_combined_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'Add-on_76_GP', 'IL1B_combined_GP', 'CDH1_ligand_receptor_target_gene_GP', 'Add-on_37_GP', 'Add-on_28_GP', 'SLPI_ligand_receptor_target_gene_GP', 'Add-on_77_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CLU_combined_GP', 'BMP5_combined_GP', 'TNXB_ligand_receptor_target_gene_GP', 'TFF1_combined_GP', 'TNFSF13B_combined_GP', 'CCL11_combined_GP', 'Add-on_70_GP', 'gamma-Aminobutyric acid_metaboli

In [47]:
adata.obs['latent_leiden_0.4'].value_counts()

latent_leiden_0.4
0     31855
1     26695
2     25249
3     22345
4     22266
5     20983
6     20849
7     18868
8     17103
9     16214
10    13495
11    13233
12    12231
13     9477
14     1953
15     1221
Name: count, dtype: int64

### Load ontology mapping file

In [48]:
# rename MTRNR2L12+ASS1+_SC to stem cells
adata.obs["Cell_State"] = adata.obs["Cell_State"].replace({"MTRNR2L12+ASS1+_SC": "stem cells"})
adata.obs["Cell_State"] = adata.obs["Cell_State"].replace({"RPS10+_RPS17+_SC": "stem cells"})
adata.obs["Cell_State"] = adata.obs["Cell_State"].replace({"FXYD3+_CKB+_SC": "stem cells"})

/var/folders/8j/syfvxwvd4lggpmnnpd_r8jkr0000gp/T/ipykernel_37114/1990493243.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs["Cell_State"] = adata.obs["Cell_State"].replace({"MTRNR2L12+ASS1+_SC": "stem cells"})
/var/folders/8j/syfvxwvd4lggpmnnpd_r8jkr0000gp/T/ipykernel_37114/1990493243.py:3: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs["Cell_State"] = adata.obs["Cell_State"].replace({"RPS10+_RPS17+_SC": "stem cells"})
/var/folders/8j/syfvxwvd4lggpmnnpd_r8jkr0000gp/T/ipykernel_37114/1990493243.py:4: FutureWarning: The behavior of Series.repl

In [49]:
adata.obs["Cell_State"].value_counts()

Cell_State
Myofibroblasts          41752
Fibroblasts             33793
Enterocyte              31886
Colonocyte              20703
Plasma cells            18559
stem cells              14871
Macrophages             14387
Goblet cells            12619
B cells                 11914
CD4 T                   11635
Glial cells              7929
Endothelial cells        7098
TA                       6115
BEST4+ epithelial        5850
Pericytes                5828
Tuft cells               4808
Mast cells               3722
EECs                     3694
DC                       3409
Arterial capillary       3242
Mature venous EC         2561
CD8 T                    2143
Mature arterial EC       1991
Monocytes                1732
LEC                       774
NK                        481
Immune Cycling cells      191
Deep_crypt_secretory      103
Lymphatics                 99
Microfold cell             95
Tregs                      28
ILCs                       18
Gamma delta T cells         6

In [50]:
with open('ontology_terms.json', 'r') as f:
    ontology_data = json.load(f)

cell_type_mapping = {item['cell_annotation']: item['ontology_term'] 
                     for item in ontology_data['ontology_terms']}

In [51]:
# Map cell_states to cell_type_ontology_term_id
adata.obs['cell_type_ontology_term_id'] = 'unknown'
for cell_type in adata.obs['Cell_State'].unique():
    if cell_type in cell_type_mapping:
        ontology_id = cell_type_mapping[cell_type]
        mask = adata.obs['Cell_State'] == cell_type
        adata.obs.loc[mask, 'cell_type_ontology_term_id'] = ontology_id
    else:
        print(f"No mapping found for '{cell_type}', keeping as 'unknown'")

No mapping found for 'EECs', keeping as 'unknown'
No mapping found for 'Deep_crypt_secretory', keeping as 'unknown'
No mapping found for 'Microfold cell', keeping as 'unknown'


In [52]:
adata.obs['organism'] = 'Homo sapiens'
adata.obs['organism_ontology_term_id'] = 'NCBITaxon:9606'

adata.obs['tissue_ontology_term_id'] = "UBERON:0000059"
adata.obs['sex_ontology_term_id'] = 'unknown'
adata.obs['assay_ontology_term_id'] = "EFO:0022615"
adata.obs['self_reported_ethnicity_ontology_term_id'] = 'unknown'
adata.obs['disease_ontology_term_id'] = 'PATO:0000461'

In [53]:
required_cols = ['organism_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 
                'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id',
                'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id']
missing_cols = [col for col in required_cols if col not in adata.obs.columns]
print(f"WARNING: Missing required columns: {missing_cols}")


In [54]:
adata.var["gene_symbol"] = adata.var.index.copy()

### Preprocessor module
scPRINT converts the gene expression of a cell to an embedding by summing three representations or tokens: its id, expression, and genomic location scdataloader. The Preprocessor module specifically handles:

Gene ID Encoding: scPRINT encodes the gene IDs using protein embeddings. This gene representation is made using the ESM2 amino-acid embedding of its most common protein product.   
Expression Embedding: The gene's expression is tokenized via a multi-layer perceptron (MLP) using log-normalized counts. This MLP lets the model learn a metric behind gene expression.   
Positional Encoding: We help the model know that genes with similar locations tend to be regulated by identical DNA regions, using the positional encoding of their location in the genome.   

The preprocessor makes sure that the fields are all here, the genes are in the right format, the raw counts are used.

In [55]:
preprocessor = Preprocessor(
    do_postp=True, # takes too much RAM in collab to do PCA
    # additional_postprocess=additional_postprocess,
    additional_preprocess=additional_preprocess,
    force_preprocess=True,
)

In [1]:
adata = preprocessor(adata)

NameError: name 'preprocessor' is not defined

In [ ]:
file = ln.Artifact.from_anndata(adata, description="test anndata")
file.save()
col = ln.Collection(file, name="test")
col.save()

### Loading the model

In [ ]:
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set environment variables to disable SSL verification
os.environ['PYTHONHTTPSVERIFY'] = '0'
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_VERIFY'] = 'false'

# Configure SSL context
ssl._create_default_https_context = ssl._create_unverified_context

# Configure requests to not verify SSL
import requests.packages.urllib3.util.ssl_
requests.packages.urllib3.util.ssl_.DEFAULT_CIPHERS = 'ALL'

# Monkey patch requests to disable SSL verification
original_request = requests.Session.request
def patched_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return original_request(self, method, url, **kwargs)
requests.Session.request = patched_request

# Now download the model
model_checkpoint_file = hf_hub_download(
    repo_id="jkobject/scPRINT", 
    filename="v2-medium.ckpt"
)

In [ ]:
model_checkpoint_file 

In [ ]:
# make sure that you check if you have a GPU with flashattention or not (see README)
try:
    m = torch.load(model_checkpoint_file)
# if not use this instead since the model weights are by default mapped to GPU types
except RuntimeError: 
    m = torch.load(model_checkpoint_file, map_location=torch.device('cpu'))

In [ ]:
# again here by default the model was trained with flash attention, so if you do not have a GPU you will need to replace the attention mechanism with regular attention 
transformer = "flash" if torch.cuda.is_available() else "normal"

In [ ]:
# both are for compatibility issues with different versions of the pretrained model, so we need to load it with the correct transformer
if "prenorm" in m['hyper_parameters']:
    m['hyper_parameters'].pop("prenorm")
    torch.save(m, model_checkpoint_file)
if "label_counts" in m['hyper_parameters']:
    # you need to set precpt_gene_emb=None otherwise the model will look for its precomputed gene embeddings files although they were already converted into model weights, so you don't need this file for a pretrained model
    model = scPrint.load_from_checkpoint(model_checkpoint_file, precpt_gene_emb=None, classes=m['hyper_parameters']['label_counts'], transformer=transformer)
else:
    model = scPrint.load_from_checkpoint(model_checkpoint_file, precpt_gene_emb=None, transformer=transformer)

In [ ]:
# this might happen if you have a model that was trained with a different set of genes than the one you are using in the ontology (e.g. newer ontologies), While having genes in the onlogy not in the model is fine. the opposite is not, so we need to remove the genes that are in the model but not in the ontology
missing = set(model.genes) - set(load_genes(model.organisms).index)
if len(missing) > 0:
    print(
        "Warning: some genes missmatch exist between model and ontology: solving...",
    )

In [ ]:
# again if not on GPU you need to convert the model to float64
if not torch.cuda.is_available():
    model = model.to(torch.float32)
    
# you can perform your inference on float16 if you have a GPU, otherwise use float64
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# the models are often loaded with some parts still displayed as "cuda" and some as "cpu", so we need to make sure that the model is fully on the right device 
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

## Download data and filter

In [ ]:
#sc.pp.filter_genes(adata, min_cells=1)

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(adata, 
                            n_top_genes=3000, 
                            batch_key="sample_id",
                            layer="counts",
                            flavor="seurat_v3",
                            subset=False,  # Don't subset - just mark highly variable genes
                            span=1)

# Let's see how many highly variable genes we have
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")
print(f"Total genes: {adata.n_vars}")

In [ ]:
# Keep the filtered dataset for GRN inference
print(f"Filtered dataset shape: {adata.shape}")
print(f"Number of highly variable genes: {adata.n_vars}")

### Gene network inference

Finally we will use scPRINT to infer gene networks on another cell of interest, the fibroblasts, in both normal and BPH conditions.

**• "most var within"**:  
    ◦ Purpose: This method identifies genes that show the highest variability or dispersion in their expression levels within a specified cell type or a homogeneous subset of your data. It aims to capture the intrinsic heterogeneity or dynamic range of gene expression within a particular biological population.
        ▪ The code first subsets the AnnData object (adata) to only include cells of the cell_type specified (if provided) into subadata.   
        ▪ It then applies sc.pp.highly_variable_genes(subadata, flavor="seurat_v3", n_top_genes=self.num_genes). This function is designed to identify genes that are highly variable within the input data subset (subadata). The seurat_v3 flavor is a common method in single-cell analysis for this purpose.   
        ▪ self.curr_genes is then populated with these identified highly variable genes.   

**• "most var across"**:   
    ◦ Purpose: This method identifies genes that are most differentially expressed or show significant variability between different groups or cell types. Its goal is to highlight genes that distinguish one cell type or condition from others, indicating potential drivers of cell identity or disease states.  
        ▪ This option explicitly checks if cell_type is not None, indicating its intention to compare a specific cell type against others (or the rest of the dataset).   
        ▪ It uses sc.tl.rank_genes_groups(adata, ..., groupby=self.cell_type_col, groups=[cell_type]). The rank_genes_groups function is a standard Scanpy tool for differential expression analysis, which inherently calculates genes that are significantly different across the defined groups (in this case, [cell_type] vs. all other cells in adata or other groups if configured).   
        ▪ diff_expr_genes are extracted from the results of this differential expression analysis, representing genes that are highly variable between the cell_type of interest and other cells.    
        ▪ self.curr_genes is then set to these top differentially expressed genes.   

In [ ]:
grn_inferer = GNInfer(
                    # here we use the most variable genes - this will automatically select from highly variable genes
                    how="most var across",  # corrected to "most var" based on the actual error message
                    # we will preprocess the attention matrix with softmax
                    preprocess="softmax",
                    # we don't aggregate the heads here, we will do it manually
                    head_agg='mean',
                    # we will not use any filtration here, but you can use "none" or "topk" for top k connections per genes or "thresh" for a defined threshold, and more
                    filtration="thresh",
                    # here if we generate the attention matrices by performing a task, like denoising or by just passing the expression profile through the model
                    forward_mode="none",
                    # use the 3000 highly variable genes
                    num_genes=10000,  # this will select from the marked highly variable genes
                    doplot=True,
                    batch_size=8,
                    cell_type_col = "cell_states",
                    # disable multiprocessing to avoid collator issues
                    num_workers=0,  # set to 0 to avoid multiprocessing errors,
                    dtype=dtype,
                    # list of layers to use
                    layer=list(range(model.nlayers))[:]
                    )

In [ ]:
# Use this dataset for GRN inference
grn_full = grn_inferer(model, adata, cell_type="Stem cells")

In [ ]:
grn_full.var

In [ ]:
grn_full.write_h5ad("1_scPRINT/data/grn_sc_adult_10K_genes.h5ad")